# scEEMS Prediction

Apply a trained feature-weighted CatBoost scEEMS model to score genetic variants for their probability of having a functional, cell-type-specific regulatory impact on gene expression. This worker runs the `predict` subcommand of `gems_pipeline.py` and emits per-variant Expression Modifier Scores.

**Author:** Angelina Siagailo, with input from Gao Wang and Katie Cho. Method and model are described in the scEEMS manuscript; the upstream training step is documented in [EMS Training](https://statfungen.github.io/xqtl-protocol/code/xqtl_modifier_score/ems_training.html).

## Description

The trained CatBoost classifier assigns each variant a continuous EMS score in `[0, 1]` representing the probability that it modifies gene expression in the target cell type. Suggested prioritization tiers: score > 0.8 high (prioritize for experimental validation such as CRISPR screens or luciferase assays); 0.5-0.8 moderate (context-dependent); < 0.5 low. New cohorts / cell types are supported by editing the YAML configs only - no Python code changes are required (e.g. swap `protocol_example` for `Ast_mega_eQTL`).

**Timing:** ~varies on typical compute infrastructure.

## Input

| File | Description |
|------|-------------|
| `--model_path` | Trained feature-weighted CatBoost model (`.joblib`) produced by the [EMS Training](https://statfungen.github.io/xqtl-protocol/code/xqtl_modifier_score/ems_training.html) workflow for the chosen cohort / chromosome. |
| `--data_config` (`data_config.yaml`) | Data configuration: cohort, paths to the variant list to score, and the full 4,839-feature set. New cohorts/cell types are added by editing this YAML only. |
| `cohort` (positional) | Cohort / cell type, e.g. `protocol_example`. Must match an entry in the data config. |
| `chromosome` (positional) | Chromosome to score, e.g. `2`. |

The variant list referenced by the data config is a tab-separated file of `variant_id` entries in `chr:pos:ref:alt` notation (1-based GRCh38 coordinates, ACGT alleles), e.g. `2:12345:A:T`.

## Output

| File | Description |
|------|-------------|
| `predictions_weighted_model_chr<chromosome>.tsv` | Per-variant predictions: `variant_id`, the continuous EMS score column (`standard_subset_weighted_pred_prob`), and a binary label column. |

The continuous score column is the primary result used to rank and prioritize variants for downstream experimental validation.

## Steps

**Step 1.** Score variants with a trained feature-weighted CatBoost scEEMS model for one cohort / chromosome. The toy command below scores the microglia cohort (`protocol_example`) on chromosome 2 using a model produced by the training workflow.

In [ ]:
import pandas as pd
import joblib
import pickle
import numpy as np

# Load trained CatBoost model
MODEL_PATH = "../../data/Mic_mega_eQTL/model_results/model_standard_subset_weighted_chr_chr2_NPR_10.joblib"
model = joblib.load(MODEL_PATH)

print("Model Configuration:")
print(f"  Algorithm: {model.__class__.__name__}")
print(f"  Features: {model.feature_count_}")
print(f"  Classes: {list(model.classes_)}")
print(f"  Tree depth: {model.get_params()['depth']}")
print(f"  Iterations: {model.tree_count_}")

# Load prediction results
RESULTS_PATH = "../../data/Mic_mega_eQTL/model_results/predictions_parquet_catboost/predictions_weighted_model_chr2.tsv"
results = pd.read_csv(RESULTS_PATH, sep='\t')

print(f"\nLoaded predictions for {len(results)} variants")
print(f"\nAvailable columns:")
print(f"  - variant_id: Genomic coordinates")
print(f"  - standard_subset_weighted_pred_prob: EMS functional score (0-1)")
print(f"  - standard_subset_weighted_pred_label: Binary classification (0/1)")
print(f"  - actual_label: True label from test set (if available)")

## Command interface

In [ ]:
score_col = 'standard_subset_weighted_pred_prob'

# Quantitative distribution analysis
print("SCORE DISTRIBUTION ANALYSIS")
print("=" * 50)
print(f"Total variants analyzed: {len(results)}")
print(f"\nPriority Classification:")
high = len(results[results[score_col] > 0.8])
medium = len(results[(results[score_col] > 0.5) & (results[score_col] <= 0.8)])
low = len(results[results[score_col] <= 0.5])
print(f"  High priority (>0.8):     {high:5d} ({high/len(results)*100:5.1f}%)")
print(f"  Medium priority (0.5-0.8): {medium:5d} ({medium/len(results)*100:5.1f}%)")
print(f"  Low priority (<0.5):      {low:5d} ({low/len(results)*100:5.1f}%)")

# Descriptive statistics
print(f"\nScore Statistics:")
print(f"  Mean:   {results[score_col].mean():.4f}")
print(f"  Median: {results[score_col].median():.4f}")
print(f"  Std:    {results[score_col].std():.4f}")
print(f"  Min:    {results[score_col].min():.4f}")
print(f"  Max:    {results[score_col].max():.4f}")

# Percentile distribution
print(f"\nPercentile Distribution:")
for p in [90, 95, 99]:
    val = np.percentile(results[score_col], p)
    count = len(results[results[score_col] >= val])
    print(f"  {p}th percentile: {val:.4f} ({count} variants)")

## Workflow implementation

The `predict` step wraps the `gems_pipeline.py predict` engine. The pipeline loads the trained model, annotates the input variants with the full feature set, applies the 10x deep-learning feature weighting used at training time, and writes per-variant EMS scores. New datasets / cell types are scored by editing the `data_config` YAML only - no Python code changes are required.

## Anticipated Results

The pipeline produces output files in the `output/` subdirectory named after the workflow step. Verify success by checking that output files exist and are non-empty. See the **Output** section above for the expected file names and formats.

In [ ]:
# Identify and sort high-priority variants
high_priority = results[results[score_col] > 0.8].sort_values(score_col, ascending=False)

print(f"HIGH-PRIORITY VARIANTS (Score >0.8)")
print("=" * 50)
print(f"Total: {len(high_priority)} variants")

if len(high_priority) > 0:
    print(f"\nTop 10 by EMS score:")
    display_cols = ['variant_id', score_col]
    if 'actual_label' in results.columns:
        display_cols.append('actual_label')
    print(high_priority[display_cols].head(10).to_string(index=False))
    
    # Export for experimental design
    output_file = "high_priority_variants_validation.tsv"
    high_priority.to_csv(output_file, sep='\t', index=False)
    print(f"\nExported to: {output_file}")
    print(f"   Contains all {len(high_priority)} high-priority variants")
    print(f"   Ready for: CRISPR screens, luciferase assays, functional validation")
else:
    print("\nNo variants exceed 0.8 threshold")
    print("   Recommended actions:")
    print("   1. Review medium-priority variants (0.5-0.8)")
    print("   2. Consider lowering threshold based on experimental capacity")
    print(f"   3. Top variant score: {results[score_col].max():.4f}")

In [ ]:
# Load and display performance metrics
SUMMARY_PATH = "../../data/Mic_mega_eQTL/model_results/summary_dict_catboost_weighted_model_chr_chr2_NPR_10.pkl"
with open(SUMMARY_PATH, 'rb') as f:
    summary = pickle.load(f)

print("MODEL PERFORMANCE ON TEST SET")
print("=" * 50)
metrics = summary['CatBoost']['standard_subset_weighted']
print(f"Average Precision (AP): {metrics['AP_test']:.4f}")
print(f"AUC-ROC:                {metrics['AUC_test']:.4f}")

print(f"\nTest Set Composition:")
print(f"  Positive labels (functional eQTLs): {summary['CatBoost']['test_num_positive_labels']}")
print(f"  Negative labels (non-functional):   {summary['CatBoost']['test_num_negative_labels']}")
pos_rate = summary['CatBoost']['test_num_positive_labels'] / (summary['CatBoost']['test_num_positive_labels'] + summary['CatBoost']['test_num_negative_labels'])
print(f"  Positive rate:                       {pos_rate:.1%}")

print("\n These metrics reflect performance on 761 held-out chromosome 2 variants")
print("   with stricter selection criteria (PIP >0.9 for positives) than training data.")